# SAX CMR: extract mid-cavity slice as 2D+t NIfTI

Reads every `*.nii.gz` / `*.nii` in `INPUT_DIR`, assumes shape `(X, Y, Z, T)`,
takes the geometric middle slice `z = Z // 2`, and writes it to `OUTPUT_DIR`
with the affine shifted so the slice keeps its true world position.

Requires: `pip install nibabel numpy`

In [4]:
from pathlib import Path
import os
import numpy as np
import nibabel as nib
from ipyfilechooser import FileChooser

In [5]:
path_chooser_dataset = FileChooser(os.path.join(os.getcwd(),''), '')
display(path_chooser_dataset)

FileChooser(path='/mnt/ssd/git/test/cmr-multi-view-phase-detection/notebooks/Dataset', filename='', title='HTM…

In [6]:
INPUT_DIR  = os.path.join(path_chooser_dataset.selected, 'sax')      # folder with 3D+t SAX volumes
OUTPUT_DIR =  os.path.join(path_chooser_dataset.selected, 'sax-mid-slice')   # created if missing

# True  -> output shape (X, Y, 1, T), a 4D NIfTI with a singleton slice axis
# False -> output shape (X, Y, T), a true 3D NIfTI where the 3rd axis is time
KEEP_SINGLETON_Z = True

SUFFIX    = "_midslice"
OVERWRITE = True

In [7]:
def extract_mid_slice(in_path, out_dir, keep_singleton_z=True, suffix="_midslice"):
    """Save the middle slice of a 3D+t NIfTI as 2D+t. Returns (out_path, z, nz)."""
    img = nib.load(str(in_path))
    if img.ndim != 4:
        raise ValueError(f"expected 4D (X, Y, Z, T), got shape {img.shape}")

    nz = img.shape[2]
    z = nz // 2
    data = np.asanyarray(img.dataobj)[:, :, z, :]          # (X, Y, T)
    if keep_singleton_z:
        data = data[:, :, None, :]                          # (X, Y, 1, T)

    # shift the origin to the chosen slice so world coordinates stay correct
    aff = img.affine.copy()
    aff[:3, 3] = (img.affine @ np.array([0.0, 0.0, z, 1.0]))[:3]

    out = nib.Nifti1Image(data, aff, header=img.header.copy())
    out.header.set_data_dtype(img.header.get_data_dtype())
    zx, zy, zz, zt = img.header.get_zooms()[:4]
    out.header.set_zooms((zx, zy, zz, zt) if keep_singleton_z else (zx, zy, zt))

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(in_path).name
    for ext in (".nii.gz", ".nii"):
        if stem.endswith(ext):
            stem = stem[: -len(ext)]
            break
    out_path = out_dir / f"{stem}{suffix}.nii.gz"
    nib.save(out, str(out_path))
    return out_path, z, nz

In [8]:
files = sorted(p for p in Path(INPUT_DIR).iterdir()
               if p.name.endswith((".nii", ".nii.gz")))
print(f"{len(files)} file(s) found in {INPUT_DIR}\n")

200 file(s) found in /mnt/ssd/data/01-3-MnMs-2/imported_SAX_3D_split/training/sax



In [9]:
ok, skipped, failed = 0, 0, 0
for f in files:
    out_path = Path(OUTPUT_DIR) / (f.name.replace(".nii.gz", "").replace(".nii", "") + SUFFIX + ".nii.gz")
    if out_path.exists() and not OVERWRITE:
        print(f"skip (exists) {f.name}")
        skipped += 1
        continue
    try:
        out_path, z, nz = extract_mid_slice(f, OUTPUT_DIR, KEEP_SINGLETON_Z, SUFFIX)
        print(f"{f.name}: slice {z}/{nz - 1} -> {out_path.name}  {nib.load(str(out_path)).shape}")
        ok += 1
    except Exception as e:
        print(f"FAILED {f.name}: {e}")
        failed += 1

print(f"\ndone: {ok} written, {skipped} skipped, {failed} failed")

001_SA_CINE.nii.gz: slice 6/11 -> 001_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
002_SA_CINE.nii.gz: slice 5/9 -> 002_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
003_SA_CINE.nii.gz: slice 5/10 -> 003_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
004_SA_CINE.nii.gz: slice 5/10 -> 004_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
005_SA_CINE.nii.gz: slice 6/11 -> 005_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
006_SA_CINE.nii.gz: slice 5/9 -> 006_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
007_SA_CINE.nii.gz: slice 5/10 -> 007_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
008_SA_CINE.nii.gz: slice 6/11 -> 008_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
009_SA_CINE.nii.gz: slice 5/10 -> 009_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
010_SA_CINE.nii.gz: slice 5/10 -> 010_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
011_SA_CINE.nii.gz: slice 5/10 -> 011_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
012_SA_CINE.nii.gz: slice 5/10 -> 012_SA_CINE_midslice.nii.gz  (256, 256, 1, 25)
013_SA_CINE.nii.gz: slice 7/13